In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pandas as pd
import os

# List of specific filenames to load
file_names = [
    '1_qwen2-0_5b_Q4_K_M.csv',
    '2_qwen2.5-1.5b_Q4_K_M.csv',
    '3_phi-2_Q4_K_M.csv',
    '4_qwen2.5-3b_Q4_K_M.csv',
    '5_OLMoE_Q4_K_M.csv',
    '6_qwen2.5-7b_Q4_K_M.csv',
    '7_llama_Q4_K_M.csv',
    '8_gemma_Q4_K_M.csv',
    '9_qwen2-0_5b_IQ4_XS.csv',
    '10_qwen2.5-1.5b_IQ4_XS.csv',
    '11_phi-2_IQ4_XS.csv',
    '12_qwen2.5-3b_IQ4_XS.csv',
    '13_OLMoE_IQ4_XS.csv',
    '14_qwen2.5-7b_IQ4_XS.csv',
    '15_llama_IQ4_XS.csv',
    '16_gemma_IQ4_XS.csv'
]

# Container for the loaded DataFrames
dfs = []

for file_name in file_names:
    # Construct the full path
    file_path = os.path.join('csv', file_name)
    
    # Read the CSV and append to the list
    dfs.append(pd.read_csv(file_path))

# Access individual frames by index (e.g., dfs[0] corresponds to df1)

In [3]:
# List of column names to remove
cols_to_remove = ['__run_id', '__done', 'model_response',
                  'battery_capacity', 'min_temperature', 'max_temperature']

# Apply drop to every dataframe in the list
dfs = [df.drop(columns=cols_to_remove, errors='ignore') for df in dfs]

In [4]:
# ... (imports and dfs loading remain the same)

aggregated_rows = []

cols_to_median = ['input_token_count', 'output_token_count',
                  'total_token_count', 'prompt_prefill_speed',
                  'generation_decoder_speed', 'prefill_latency',
                  'generation_latency', 'inference_latency',
                  'time_to_first_token','avg_current',
                  'avg_voltage','avg_power',
                  'total_energy_consumption','energy_per_token',
                  'average_temperature',
                  'peak_memory','model_weight','KV_cache',
                  'context_RAM','compute_RAM'] 


cols_to_IQR = [ 'prompt_prefill_speed','generation_decoder_speed',
                'prefill_latency','generation_latency',
                'inference_latency','time_to_first_token',
                'avg_power','total_energy_consumption','energy_per_token'] 

# Loop through all loaded dataframes
for df in dfs:
    if df.empty:
        continue
        
    # 1. Calculate basic Median
    model_name_val = df['model_file'].iloc[0]
    median_series = df[cols_to_median].median()
    
    # --- FIX STARTS HERE ---
    # Convert to object type so pandas allows us to overwrite numbers with strings
    median_series = median_series.astype(object)
    # --- FIX ENDS HERE ---

    # 2. Calculate IQR
    Q1 = df[cols_to_IQR].quantile(0.25)
    Q3 = df[cols_to_IQR].quantile(0.75)
    iqr_series = Q3 - Q1
    
    # 3. Merge IQR into the Median value as a string
    for col in cols_to_IQR:
        # We can now safely insert strings because we cast to object above
        median_series[col] = f"{median_series[col]:.2f} (IQR {iqr_series[col]:.2f})"

    # 4. Calculate battery usage
    battery_val = (df['max_battery_capacity'].max() - df['min_battery_capacity'].min()) / 30

    # 5. Add metadata
    single_row = median_series.copy()
    single_row['model_name'] = model_name_val
    single_row['battery_usage'] = battery_val
    
    aggregated_rows.append(single_row)

# Create final DataFrame
final_df = pd.DataFrame(aggregated_rows)

# Apply the column order
idx = cols_to_median.index('energy_per_token') + 1
new_order = (
    ['model_name'] + 
    cols_to_median[:idx] + 
    ['battery_usage'] + 
    cols_to_median[idx:]
)

final_df = final_df[new_order]

In [ ]:
final_df.to_csv('final_results.csv', index=False)